# COMP7103 Lab 2 · Search product reviews two ways

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab2/Lab2_search.ipynb)

You get 500 short product reviews and their **embeddings, computed in advance**. For 5 queries, you compare the top-5 hits from **TF-IDF + cosine** with the top-5 hits from **embeddings + cosine**, judge which hits are relevant, and score both methods.

| Time | What you do |
|---|---|
| 5 min | Brief: read the data and the 5 queries |
| 30 min | Run both methods, judge the hits, compute precision@5 |
| 10 min | Share-out |

This lab is ungraded practice for the exams.

*Data: 500 sentences from the Amazon part of Kotzias (2015), Sentiment Labelled Sentences, UCI Machine Learning Repository, https://doi.org/10.24432/C57604, CC BY 4.0. Embeddings: all-MiniLM-L6-v2. See data/SOURCE.txt.*

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup

This cell loads the reviews, the queries and the embeddings. Nothing needs installing.

In [ ]:
import os, re, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 160)

# A blank (___) that you have not filled in stops with: name '___' is not defined
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab2/data/"

def get(name):
    """Use the local file if it is there (Lab 2 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 2 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab2")
    return os.path.join("data", name)

reviews = pd.read_csv(get("reviews.csv"))
queries = pd.read_csv(get("queries.csv"))
review_emb = np.load(get("review_embeddings.npy"))
query_emb = np.load(get("query_embeddings.npy"))
print(len(reviews), "reviews ·", len(queries), "queries · embeddings", review_emb.shape)

## 1 · The data (5 min)

Each review is one sentence about a phone or an accessory. `sentiment` is 1 for positive, 0 for negative (from the source data; you will not need it for scoring).

In [ ]:
reviews.sample(5, random_state=1)

The 5 queries. `relevant_if` is the rule you use to judge each hit.

In [ ]:
queries

## 2 · TF-IDF + cosine (5 min)

The weights from Lecture 2:  **TF<sub>d</sub>(t) × IDF(t)**, with **IDF(t) = log(|D| / |D<sub>t</sub>|)**, where |D| is the number of documents and |D<sub>t</sub>| the number of documents containing t. We use log base 10.

First, the dictionary and the term-frequency matrix (given):

In [ ]:
STOP = set("a an the on in of and or is are was were be to it this that with for but my at by as i its".split())

def tokens(text):
    """Lowercase, split into words (hyphenated words stay one word), drop stop words."""
    return [w for w in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if w not in STOP]

docs = [tokens(t) for t in reviews["text"]]
vocab = sorted({w for d in docs for w in d})
index = {w: i for i, w in enumerate(vocab)}

def counts(words):
    """Term-frequency vector over the dictionary (words not in the dictionary are ignored)."""
    v = np.zeros(len(vocab))
    for w in words:
        if w in index:
            v[index[w]] += 1
    return v

TF = np.array([counts(d) for d in docs])
print(TF.shape, "= reviews × terms;", int((TF > 0).sum()), "non-zero entries")

**Check A.** Fill in |D|, the number of reviews.

In [ ]:
df_t = (TF > 0).sum(axis=0)        # |D_t|: number of reviews containing each term
idf = np.log10(___ / df_t)        # IDF(t) = log(|D| / |D_t|)
W = TF * idf                       # TF-IDF weights
print("IDF of \"phone\":", round(idf[index["phone"]], 3), "· IDF of \"nokia\":", round(idf[index["nokia"]], 3))

**Check B.** Cosine similarity, as in Lecture 2: cos(d<sub>1</sub>, d<sub>2</sub>) = (d<sub>1</sub> • d<sub>2</sub>) / (||d<sub>1</sub>|| ||d<sub>2</sub>||). Fill in the dot product of every review vector (the rows of `A`) with the query vector `b`.

In [ ]:
def cosine(A, b):
    """Cosine similarity between every row of A and the vector b (0 if a vector is all zeros)."""
    norms = np.linalg.norm(A, axis=1) * np.linalg.norm(b)
    dots = A @ ___
    return np.where(norms > 0, dots / np.where(norms > 0, norms, 1), 0.0)

def tfidf_scores(query):
    return cosine(W, counts(tokens(query)) * idf)

tfidf_scores("Nokia").max()

## 3 · Embeddings + cosine (3 min)

Each review is already a vector of 384 numbers, and so is each query. The vectors have length 1, so their cosine similarity is just the dot product.

**Check C.** Fill in the query's embedding: row `k` of `query_emb`.

In [ ]:
print("lengths of the first three review vectors:", np.linalg.norm(review_emb[:3], axis=1).round(3))

def emb_scores(k):
    """Cosine similarity between every review and row k of queries (k = 0 is query 1, ..., k = 4 is query 5, Nokia)."""
    return review_emb @ ___

emb_scores(4).max()

## 4 · Compare the top 5 (2 min)

This cell lists the top-5 hits of both methods for every query (given). A TF-IDF score of 0 means the review shares no term with the query.

In [ ]:
def top5(scores):
    return np.argsort(-scores, kind="stable")[:5]

def show(k):
    q = queries.loc[k]
    t, e = top5(tfidf_scores(q["query"])), top5(emb_scores(k))
    print(f'Query {q.query_id}: "{q.query}"  ·  relevant if: {q.relevant_if}')
    return pd.DataFrame({
        "rank": range(1, 6),
        "TF-IDF": [f"{tfidf_scores(q['query'])[i]:.2f}  {reviews.text[i]}" for i in t],
        "Embeddings": [f"{emb_scores(k)[i]:.2f}  {reviews.text[i]}" for i in e],
    }).set_index("rank")

for k in range(len(queries)):
    display(show(k))

## 5 · Judge the hits and score them (15 min)

For every query, read the 10 hits above. In the cell below, change a 0 to **1** for each relevant hit, in the order shown (rank 1 to 5). Use the `relevant_if` rule.

**Precision@5** = number of relevant hits in the top 5 / 5.

In [ ]:
judgments = {
    1: {"tfidf": [0, 0, 0, 0, 0], "emb": [0, 0, 0, 0, 0]},   # battery dies too quickly
    2: {"tfidf": [0, 0, 0, 0, 0], "emb": [0, 0, 0, 0, 0]},   # easy to set up
    3: {"tfidf": [0, 0, 0, 0, 0], "emb": [0, 0, 0, 0, 0]},   # broke after a few days
    4: {"tfidf": [0, 0, 0, 0, 0], "emb": [0, 0, 0, 0, 0]},   # not worth the money
    5: {"tfidf": [0, 0, 0, 0, 0], "emb": [0, 0, 0, 0, 0]},   # Nokia
}

In [ ]:
p5 = pd.DataFrame({
    "query": queries["query"].values,
    "TF-IDF relevant": [sum(judgments[k]["tfidf"]) for k in queries.query_id],
    "TF-IDF P@5": [sum(judgments[k]["tfidf"]) / 5 for k in queries.query_id],
    "Embeddings relevant": [sum(judgments[k]["emb"]) for k in queries.query_id],
    "Embeddings P@5": [sum(judgments[k]["emb"]) / 5 for k in queries.query_id],
})
display(p5.round(2))
print("Mean precision@5 · TF-IDF:", p5["TF-IDF P@5"].mean().round(2), "· embeddings:", p5["Embeddings P@5"].mean().round(2))
p5.set_index("query")[["TF-IDF P@5", "Embeddings P@5"]].plot.bar(rot=20, ylim=(0, 1), figsize=(8, 3.5), title="Precision@5", ylabel="precision@5")
plt.tight_layout(); plt.show()

## 6 · Where does each method fail? (5 min)

Double-click this cell to edit it. Write one note per method: a query where it failed, and why.

| Method | Query | What went wrong | Why |
|---|---|---|---|
| TF-IDF |  |  |  |
| Embeddings |  |  |  |

## 7 · Share-out (10 min)

- For each query and each method, report how many of the top 5 hits you judged relevant (0–5). The class board turns these into precision@5.
- Where did TF-IDF win, and where did embeddings win?
- Which hits were hard to judge, and why?

## Optional · Try your own query

TF-IDF works for any query right away. Embeddings need the model to encode a new query: uncomment the lines below (this downloads the model, about 90 MB).

In [ ]:
my_query = "great sound quality"
scores = tfidf_scores(my_query)
display(pd.DataFrame({"score": scores[top5(scores)].round(2), "review": reviews.text[top5(scores)].values}))

# from sentence_transformers import SentenceTransformer        # preinstalled in Colab
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# e = review_emb @ model.encode(my_query, normalize_embeddings=True)
# display(pd.DataFrame({"score": e[top5(e)].round(2), "review": reviews.text[top5(e)].values}))